# 02 · Link check (polite HTTP pass + real-browser fallback)

I check every **distinct** resource URL once and join the results back to every
catalogue link that uses it.

How I behave towards the servers (all set in `hclr/config.py`):
- a descriptive User-Agent that names the project;
- at most one request in flight per host, at least 2 s between requests to the same host, and at most 4 hosts at once;
- HEAD first, then one GET (headers only, body never read) only if HEAD returned an error;
- timeouts of 10 s to connect and 30 s to read, with **no retries**;
- a 429 is recorded as-is, and I then back off that host for at least 60 s;
- a **circuit breaker**: once a host has answered 10 requests in a row with a WAF challenge, I stop sending
  it requests and record the URL I'd reached. Every further request there would only ever be answered with
  "prove you're human".

Status codes and error messages are stored verbatim in the Bronze JSONL. Any
classification happens later, in notebook 03, so I can audit and change the rules
without touching a server again.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from hclr import config


In [2]:
from hclr import checker, browser
resources = pd.read_parquet(config.SILVER / "resources.parquet")
stamp = resources["snapshot_utc"].iloc[0]
LC_DIR = config.BRONZE / "linkcheck" / stamp
HTTP_OUT, BROWSER_OUT = LC_DIR / "http_results.jsonl", LC_DIR / "browser_results.jsonl"
urls = resources["url"].drop_duplicates().tolist()
print(len(urls), "distinct URLs to check ->", HTTP_OUT.relative_to(ROOT))

3111 distinct URLs to check -> data/bronze/linkcheck/2026-11-21T014501Z/http_results.jsonl


## Pass 1: HTTP

This is resumable. If it is interrupted, re-running the cell carries on from where it stopped.

In [3]:
checker.run(urls, HTTP_OUT);

3111 already checked; 0 to check across 0 hosts
finished: 0/0


In [4]:
http = pd.json_normalize(checker.load_results(HTTP_OUT))
eff_status = http["get.status"].fillna(http["head.status"]) if "get.status" in http else http["head.status"]
http["effective_status"] = eff_status
http["effective_error"] = http.get("get.error_kind", pd.Series(index=http.index)).fillna(http["head.error_kind"])
http["challenge"] = http["head.challenge"]
print(len(http), "URLs checked")
http.groupby(["effective_status", "effective_error", "challenge"], dropna=False).size().sort_values(ascending=False).head(30)

3111 URLs checked


effective_status  effective_error              challenge           
NaN               stopped_at_challenging_host  NaN                     1766
200.0             NaN                          NaN                     1201
409.0             NaN                          NaN                      102
403.0             NaN                          cloudflare:challenge      10
202.0             NaN                          aws-waf:challenge         10
405.0             NaN                          aws-waf:captcha           10
403.0             NaN                          NaN                        5
NaN               NaN                          NaN                        4
                  dns_failure                  NaN                        1
                  invalid_url                  NaN                        1
                  timeout                      NaN                        1
dtype: int64

## Pass 2: real browser, only for 403s, timeouts and WAF challenges

The brief says, and the NDL's method says, that a 403 is not a dead dataset. I send
stock headless Chromium at these URLs only, with 3 s between navigations per host.
If the first five on a host all stay challenged, I stop trying that host and record why.

In [5]:
recheck = http[(http["effective_status"] == 403) | (http["effective_error"].isin(["timeout", "stopped_at_challenging_host"]))
               | http["challenge"].notna()]
print(len(recheck), "URLs flagged for a browser re-check")
browser.run(recheck["url"].tolist(), BROWSER_OUT);

1802 URLs flagged for a browser re-check


1802 already re-checked; 0 to go across 0 hosts


## Pass 2b: one patient second visit, on hosts the browser *can* get through

Some challenge pages hadn't resolved within 6 s. Where the browser got through on at least one
other URL on the same host, the host clearly isn't refusing browsers. So I give each of those
still-challenged URLs exactly **one** more visit, with 20 s for the challenge to resolve. I don't
do this on hosts where the browser never got through at all.

In [6]:
import json
PATIENT_OUT = LC_DIR / "browser_results_patient.jsonl"
br1 = pd.DataFrame([json.loads(l) for l in BROWSER_OUT.read_text().splitlines()]) if BROWSER_OUT.exists() else pd.DataFrame(columns=["url", "host"])
if len(br1):
    passable = set(br1.loc[br1["browser_challenged"].eq(False) & br1["browser_status"].notna(), "host"])
    patient = br1[br1["browser_challenged"].eq(True) & br1["host"].isin(passable)]["url"].tolist()
    print(len(patient), "URLs get one patient second visit on", sorted(passable & set(br1.loc[br1['url'].isin(patient), 'host'])))
    if patient:
        browser.run(patient, PATIENT_OUT, settle_ms=20000)

9 URLs get one patient second visit on ['www.england.nhs.uk']
9 already re-checked; 0 to go across 0 hosts


In [7]:
br = pd.DataFrame([json.loads(l) for l in BROWSER_OUT.read_text().splitlines()]) if BROWSER_OUT.exists() else pd.DataFrame(columns=["url"])
if PATIENT_OUT.exists():
    # The second visit, where there was one, supersedes the first. I keep both raw files in Bronze.
    br2 = pd.DataFrame([json.loads(l) for l in PATIENT_OUT.read_text().splitlines()]).assign(browser_pass="patient")
    br = pd.concat([br.assign(browser_pass="first")[~br["url"].isin(br2["url"])], br2], ignore_index=True)
links = http.merge(br.drop(columns=["host", "checked_utc"], errors="ignore"), on="url", how="left")
# Lists don't round-trip nicely through Parquet with mixed nulls, so I keep them as JSON text.
for c in [c for c in links.columns if links[c].map(lambda v: isinstance(v, list)).any()]:
    links[c] = links[c].map(lambda v: json.dumps(v) if isinstance(v, list) else None)
links.to_parquet(config.SILVER / "link_checks.parquet", index=False)
print(len(links), "rows ->", (config.SILVER / "link_checks.parquet").relative_to(ROOT))
br.groupby(["browser_skipped", "browser_status", "browser_challenged"], dropna=False).size() if len(br) else "no browser re-checks" 

3111 rows -> data/silver/link_checks.parquet


browser_skipped                  browser_status  browser_challenged
host_blocks_browser_after_probe  NaN             NaN                   1713
NaN                              200.0           False                   62
                                 202.0           True                     2
                                 403.0           True                    16
                                                 False                    4
                                 405.0           True                     5
dtype: int64